# 03 · Grounded generation

Retrieval finds evidence; generation must *use only that evidence* and cite it.
This notebook runs the `GenerationService` with the deterministic local
`HeuristicLLMClient` (the default offline path) and shows how the service:

- renders an evidence-grounded prompt,
- validates that every citation refers to a retrieved chunk, and
- refuses when the context does not support an answer.

In [1]:
from pathlib import Path

from ragops_lab.generation import GenerationService, HeuristicLLMClient, PromptTemplate
from ragops_lab.ingestion import ChunkingConfig, ingest_directory, load_chunks_jsonl
from ragops_lab.retrieval import BM25Retriever

DATA = Path("../data")

chunks_path = DATA / "processed" / "chunks.jsonl"
ingest_directory(DATA / "sample_documents", chunks_path, ChunkingConfig(chunk_size=220, overlap=20))
chunks = load_chunks_jsonl(chunks_path)
retriever = BM25Retriever(chunks)
service = GenerationService(HeuristicLLMClient())

## 1. Inspect the grounded prompt

The prompt template pins the model to the retrieved contexts and a strict JSON
output contract. Seeing the rendered prompt makes the grounding explicit.

In [2]:
question = "Which Apollo mission first landed on the Moon?"
contexts = retriever.search(question, top_k=3)
print(PromptTemplate().render(question, contexts))

Answer only using the provided contexts. Return valid JSON with answer_text, citations, and refusal. Cite chunk_ids exactly.
Question: Which Apollo mission first landed on the Moon?
Contexts:
[apollo-program:0] The Apollo program landed humans on the Moon. Apollo 11 was the first mission to land astronauts on the lunar surface in July 1969. Neil Armstrong and Buzz Aldrin walked on the Moon while Michael Collins remained in luna


## 2. Generate a cited answer

`service.answer` parses the model output, rejects any citation that is not in
the retrieved set, and marks the answer `grounded` only when it is both cited
and not a refusal.

In [3]:
answer = service.answer(question, contexts, model_name="heuristic-local")
print("answer   :", answer.answer_text)
print("citations:", answer.citations)
print("grounded :", answer.grounded)
print("refusal  :", answer.refusal)

answer   : The Apollo program landed humans on the Moon
citations: ['apollo-program:0']
grounded : True
refusal  : False


## 3. Refusal when retrieval returns no evidence

When retrieval surfaces no usable context, a well-behaved RAG system should
refuse rather than hallucinate. Here we simulate an empty retrieval (e.g. a
question about a topic absent from the corpus) by passing no contexts: the
service refuses and stays ungrounded.

In [4]:
no_evidence = service.answer(
    "What is the capital of France?",
    [],
    model_name="heuristic-local",
)
print("answer  :", no_evidence.answer_text)
print("refusal :", no_evidence.refusal)
print("grounded:", no_evidence.grounded)

answer  : I do not have enough evidence to answer.
refusal : True
grounded: False


## 4. Invalid citations are rejected

Swapping in a `FakeLLMClient` that fabricates a citation shows the safety check:
the service raises rather than returning an ungrounded answer.

In [5]:
import json

from ragops_lab.generation import FakeLLMClient

rogue = GenerationService(
    FakeLLMClient(json.dumps({"answer_text": "Apollo 11.", "citations": ["does-not-exist:0"], "refusal": False}))
)
try:
    rogue.answer(question, contexts, model_name="rogue")
except ValueError as exc:
    print("rejected:", exc)

rejected: Unknown citations returned by model: ['does-not-exist:0']


Next: [`04_rag_evaluation`](04_rag_evaluation.ipynb) measures answer quality
end to end and exports a report.